In [10]:
# ---------------- Task 1: first three expansion ----------------
print("\n" + "=" * 70)
print("TASK 1: first three A* expansions (supplied h)")
r = astar_traced(graph, h, START, GOAL)
print("start: frontier = [S(g=0, h=5, f=0+5=5)]")
for i, row in enumerate(r["rows"][:3], 1):
    print(f"\nExpansion {i}: select {row['node']}  g={row['g']}, h={row['h']}, f={row['g']}+{row['h']}={row['f']}")
    for nxt, g0, w, ng, hn, fn in row["generated"]:
        print(f"   generate {nxt}: g={g0}+{w}={ng}, h={hn}, f={ng}+{hn}={fn}")
    print("   frontier after expansion:", fmt_frontier(row["frontier"]))
    print("   next selected:", row["frontier"][0][2], "(lowest f; ties -> lower g, then alphabetical)")



TASK 1: first three A* expansions (supplied h)
start: frontier = [S(g=0, h=5, f=0+5=5)]

Expansion 1: select S  g=0, h=5, f=0+5=5
   generate A: g=0+1=1, h=4, f=1+4=5
   generate B: g=0+4=4, h=4, f=4+4=8
   frontier after expansion: [A(g=1,f=5), B(g=4,f=8)]
   next selected: A (lowest f; ties -> lower g, then alphabetical)

Expansion 2: select A  g=1, h=4, f=1+4=5
   generate C: g=1+2=3, h=2, f=3+2=5
   frontier after expansion: [C(g=3,f=5), B(g=4,f=8)]
   next selected: C (lowest f; ties -> lower g, then alphabetical)

Expansion 3: select C  g=3, h=2, f=3+2=5
   generate G: g=3+3=6, h=0, f=6+0=6
   frontier after expansion: [G(g=6,f=6), B(g=4,f=8)]
   next selected: G (lowest f; ties -> lower g, then alphabetical)


In [9]:
# ---------------- Task 2: A* vs UCS ----------------
print("\n" + "=" * 70)
print("TASK 2: A* (supplied h) vs UCS (all h = 0), same graph, start S, goal G")
ra = report_run("A* with supplied heuristic", graph, h)
print()
zero = {n: 0 for n in graph}
ru = report_run("UCS control (h = 0 everywhere)", graph, zero)
print("\nUCS full trace:")
for row in ru["rows"]:
    print(f"  expand {row['node']} (g={row['g']}): generated", [(n, ng) for n, _, _, ng, _, _ in row["generated"]],
          "frontier", fmt_frontier(row["frontier"]))
print("\ncomparison | path | cost | expanded count | expanded nodes | stale")
print("A*  |", "->".join(ra["path"]), "|", path_cost(graph, ra["path"]), "|", len(ra["expanded"]), "|", ra["expanded"], "|", ra["stale"])
print("UCS |", "->".join(ru["path"]), "|", path_cost(graph, ru["path"]), "|", len(ru["expanded"]), "|", ru["expanded"], "|", ru["stale"])


TASK 2: A* (supplied h) vs UCS (all h = 0), same graph, start S, goal G
--- A* with supplied heuristic ---
processed (node, g, h, f): [('S', 0, 5, 5), ('A', 1, 4, 5), ('C', 3, 2, 5), ('G', 6, 0, 6)]
expanded nodes           : ['S', 'A', 'C'] -> count = 3
stale entries ignored    : 0
returned path            : ['S', 'A', 'C', 'G'] | returned cost: 6 | independently summed cost: 6

--- UCS control (h = 0 everywhere) ---
processed (node, g, h, f): [('S', 0, 0, 0), ('A', 1, 0, 1), ('C', 3, 0, 3), ('B', 4, 0, 4), ('G', 6, 0, 6)]
expanded nodes           : ['S', 'A', 'C', 'B'] -> count = 4
stale entries ignored    : 0
returned path            : ['S', 'A', 'C', 'G'] | returned cost: 6 | independently summed cost: 6

UCS full trace:
  expand S (g=0): generated [('A', 1), ('B', 4)] frontier [A(g=1,f=1), B(g=4,f=4)]
  expand A (g=1): generated [('C', 3)] frontier [C(g=3,f=3), B(g=4,f=4)]
  expand C (g=3): generated [('G', 6)] frontier [B(g=4,f=4), G(g=6,f=6)]
  expand B (g=4): generated [] fron

In [7]:
# ---------------- Task 3: overestimate h(C) = 10 ----------------
print("\n" + "=" * 70)
print("TASK 3: overestimate - change ONLY h(C) from 2 to 10 (true remaining cost of C is 3)")
h_over = dict(h); h_over["C"] = 10
print("h(C): original", h["C"], "| modified", h_over["C"], "| true remaining", hstar["C"], "| overestimate:", h_over["C"] > hstar["C"])
print("admissible after change? ", all(h_over[n] <= hstar[n] for n in graph))
print("consistency after change:", [f"{u}->{v}:{h_over[u] <= w + h_over[v]}" for u, l in graph.items() for v, w in l])
ro = astar_traced(graph, h_over, START, GOAL)
for i, row in enumerate(ro["rows"], 1):
    print(f"\nExpansion {i}: {row['node']} g={row['g']} h={row['h']} f={row['f']}")
    for nxt, g0, w, ng, hn, fn in row["generated"]:
        print(f"   generate {nxt}: g={g0}+{w}={ng}, h={hn}, f={fn}")
    print("   frontier after:", fmt_frontier(row["frontier"]))
print()
report_run("A* with h(C)=10", graph, h_over)
print("\nchanged vs original A*?  path:", ro["path"] != ra["path"], "| cost:", ro["cost"] != ra["cost"],
      f"| expansion count: {len(ra['expanded'])} -> {len(ro['expanded'])}")

print("\nSweep: change only h(C), keep everything else (does the answer always change?)")
print("h(C) | overestimate? | path | cost | expanded | optimal (cost = 6)?")
for hc in range(0, 13):
    hh = dict(h); hh["C"] = hc
    rr = astar_traced(graph, hh, START, GOAL)
    print(hc, hc > hstar["C"], "->".join(rr["path"]), rr["cost"], rr["expanded"], rr["cost"] == 6, sep=" | ")



TASK 3: overestimate - change ONLY h(C) from 2 to 10 (true remaining cost of C is 3)
h(C): original 2 | modified 10 | true remaining 3 | overestimate: True
admissible after change?  False
consistency after change: ['S->A:True', 'S->B:True', 'A->C:True', 'B->G:True', 'C->G:False']

Expansion 1: S g=0 h=5 f=5
   generate A: g=0+1=1, h=4, f=5
   generate B: g=0+4=4, h=4, f=8
   frontier after: [A(g=1,f=5), B(g=4,f=8)]

Expansion 2: A g=1 h=4 f=5
   generate C: g=1+2=3, h=10, f=13
   frontier after: [B(g=4,f=8), C(g=3,f=13)]

Expansion 3: B g=4 h=4 f=8
   generate G: g=4+5=9, h=0, f=9
   frontier after: [G(g=9,f=9), C(g=3,f=13)]

--- A* with h(C)=10 ---
processed (node, g, h, f): [('S', 0, 5, 5), ('A', 1, 4, 5), ('B', 4, 4, 8), ('G', 9, 0, 9)]
expanded nodes           : ['S', 'A', 'B'] -> count = 3
stale entries ignored    : 0
returned path            : ['S', 'B', 'G'] | returned cost: 9 | independently summed cost: 9

changed vs original A*?  path: True | cost: True | expansion count: 3 